In [1]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama
import os

In [2]:
# --- Model Download Setup ---
# 1. Define where you want to save the model
local_model_dir = r"D:\NetShieldAI_Chatbot\pretrained_language_model"
os.makedirs(local_model_dir, exist_ok=True)

In [3]:
# 2. Specify the model repo and the specific GGUF file
# Switching to Qwen 2.5 Coder (3B) - The best small model for coding/cybersec
model_id = "bartowski/Qwen2.5-Coder-3B-Instruct-GGUF"

# We use Q4_K_M. 
# File Size: ~1.93 GB. 
# This is ideal for your 2GB GPU (NVIDIA MX550).
filename = "Qwen2.5-Coder-3B-Instruct-Q4_K_M.gguf"

In [4]:
# 3. Download the GGUF file
print(f"Downloading {filename} from {model_id} to {local_model_dir}...")
model_path = hf_hub_download(
    repo_id=model_id,
    filename=filename,
    local_dir=local_model_dir,
    local_dir_use_symlinks=False,
    resume_download=True  # <--- ADD THIS LINE
)
print(f"Model downloaded to: {model_path}")

d:\chatbot-env\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
d:\chatbot-env\Lib\site-packages\huggingface_hub\file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Qwen2.5-Coder-3B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/1.93G [00:00<?, ?B/s]

Model downloaded to: D:\NetShieldAI_Chatbot\pretrained_language_model\Qwen2.5-Coder-3B-Instruct-Q4_K_M.gguf


In [5]:
llm = Llama(
    model_path=model_path,
    n_ctx=8192, 
    n_gpu_layers=-1,  # Change 20 to -1 to attempt offloading ALL layers
    n_batch=512,
    chat_format="chatml",
    verbose=True
)

llama_model_loader: loaded meta data with 39 key-value pairs and 434 tensors from D:\NetShieldAI_Chatbot\pretrained_language_model\Qwen2.5-Coder-3B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 Coder 3B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen2.5-Coder
llama_model_loader: - kv   5:                         general.size_label str              = 3B
llama_model_loader: - kv   6:                            general.license str              

In [6]:
# 5. Test with Qwen's specific format
print("\n--- Chat Example (Qwen Format) ---")

def generate_chat_response(llm_instance, user_input_text, max_tokens=500):
    # Qwen uses ChatML format: <|im_start|>system...
    # We can use the built-in create_chat_completion to handle this automatically
    # because we set chat_format="chatml" in the loader!
    
    response = llm_instance.create_chat_completion(
        messages=[
            {"role": "system", "content": "You are a helpful cybersecurity coding assistant."},
            {"role": "user", "content": user_input_text}
        ],
        max_tokens=max_tokens,
        temperature=0.7
    )
    return response["choices"][0]["message"]["content"]

# Test Query
user_query = "Write a Python script to scan for open ports on a local IP."
print(f"User: {user_query}")

response_text = generate_chat_response(llm, user_query)
print(f"Qwen: {response_text}")


--- Chat Example (Qwen Format) ---
User: Write a Python script to scan for open ports on a local IP.


llama_perf_context_print:        load time =     974.72 ms
llama_perf_context_print: prompt eval time =     974.02 ms /    35 tokens (   27.83 ms per token,    35.93 tokens per second)
llama_perf_context_print:        eval time =   61432.63 ms /   499 runs   (  123.11 ms per token,     8.12 tokens per second)
llama_perf_context_print:       total time =   64784.72 ms /   534 tokens
llama_perf_context_print:    graphs reused =        483


Qwen: To scan for open ports on a local IP using Python, you can use the `socket` module to create a socket and attempt to connect to each port on the specified IP. Here's a simple script that does this:

```python
import socket

def scan_ports(ip, start_port, end_port):
    open_ports = []
    for port in range(start_port, end_port + 1):
        try:
            # Create a socket object
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            # Set a timeout for the connection attempt
            sock.settimeout(1)
            # Attempt to connect to the port
            result = sock.connect_ex((ip, port))
            # If the result is 0, the port is open
            if result == 0:
                open_ports.append(port)
            # Close the socket
            sock.close()
        except socket.error:
            pass
    return open_ports

def main():
    ip = '127.0.0.1'  # Replace with the local IP address you want to scan
    start_port = 1  # Start s